# Baseline ViHateT5-base: fine-tune trên dữ liệu Voz của nhóm (3 seed)

ViHateT5 (Luan Thanh Nguyen, Findings of ACL 2024) là mô hình T5 tiếng Việt được tiền huấn luyện trên **VOZ-HSD** (~10,7 triệu bình luận Voz, gán nhãn tự động), tức là cùng nguồn với dữ liệu của nhóm. Notebook này fine-tune `tarudesu/ViHateT5-base` trên tập train của nhóm để so với PhoBERT-CNN / ViSoBERT trên **cùng tập test, cùng chỉ số**.

**Cách phân loại.** T5 sinh nhãn dạng chữ. Thay vì để mô hình sinh tự do (có thể ra chuỗi lạ), notebook **chấm điểm** ba chuỗi nhãn `clean` / `offensive` / `hate`: log-xác suất của mỗi chuỗi khi biết đầu vào. Ba điểm này đóng vai trò logits, nên temperature scaling, ECE và coverage của tầng 2 tính đúng như các mô hình khác.

**Lưu ý khi viết báo cáo.**
- Mô hình đã thấy rất nhiều văn bản Voz khi tiền huấn luyện. Nếu thắng thì một phần là nhờ lợi thế miền; một số bình luận test có thể đã có trong VOZ-HSD (bài gốc không nêu khoảng thời gian crawl).
- Huấn luyện ở fp32 mặc định: T5 dễ bị tràn số (NaN) với fp16, và GPU T4 không hỗ trợ bf16 hiệu quả.
- Kết quả được lưu cùng định dạng với notebook PhoBERT (`results_*.csv`, `speed_*.json`), nên mục so sánh của notebook nào cũng gộp được.

In [ ]:
%pip install -q transformers accelerate sentencepiece

## 0. Cấu hình

In [ ]:
import os, json, random, time, gc, hashlib, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
from scipy.optimize import minimize_scalar

DATA_DIR = "/kaggle/input/voz-toxic-clean"       # cùng dataset với notebook PhoBERT ở giai đoạn đang so sánh
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.path.abspath("."))   # tự nhận Kaggle / Colab / máy cục bộ
LABEL_FIX_PATH = os.path.join(DATA_DIR, "label_fixes.csv")

STAGE = "t5_finetune"
INPUT_MODE = "text_only"       # "text_only" | "text_quote"
CFG = {"input_mode": INPUT_MODE, "use_fixes": False, "patience": 2}   # use_fixes=False nếu file đã sửa nhãn sẵn

MODEL_NAME = "tarudesu/ViHateT5-base"
BACKBONE = "vihatet5"
TASK_PREFIX = "hate-speech-detection: "   # prefix ViHateT5 dùng cho tác vụ CLEAN/OFFENSIVE/HATE
TARGETS = ["clean", "offensive", "hate"]  # chuỗi nhãn mô hình học cách sinh (thứ tự = id nhãn)
SEEDS = [42, 13, 2024]
MAX_LEN = 256
BATCH_SIZE = 16                # OOM -> 8
EPOCHS = 4
LR = 3e-4                      # mức lr thường dùng khi fine-tune T5-base
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
USE_FP16 = False               # True nhanh hơn nhưng T5 có thể ra NaN; nếu bật mà loss = nan thì tắt lại
TARGET_ACC = 0.97
MEASURE_CPU = True

TAG = f"{BACKBONE}_{INPUT_MODE}"
LABELS = ["clean", "offensive", "hate"]
LABEL_TO_ID = {l: i for i, l in enumerate(LABELS)}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = DEVICE.type == "cuda" and USE_FP16
os.makedirs(OUT_DIR, exist_ok=True)
print("TAG:", TAG, "| Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "", "| fp16:", use_amp)

## 1. Đọc dữ liệu (giống hệt notebook PhoBERT)

In [ ]:
def read_split(name):
    d = pd.read_csv(os.path.join(DATA_DIR, f"{name}_clean.csv"))
    d["row_id"] = np.arange(len(d))          # khớp với row_id trong label_fixes.csv
    d["text"] = d["text"].fillna("").astype(str)
    d["quoted_text"] = d["quoted_text"].fillna("").astype(str)
    d["label_str"] = d["label"].astype(str).str.strip().str.lower()
    bad = ~d["label_str"].isin(LABELS)
    if bad.any(): raise ValueError(f"{name}: nhãn lạ {d.loc[bad, 'label_str'].unique()[:5]}")
    d["label"] = d["label_str"].map(LABEL_TO_ID).astype(int)
    return d.drop(columns=[c for c in ["text_raw", "quoted_raw"] if c in d.columns])

train_df, val_df, test_df = read_split("train"), read_split("val"), read_split("test")

if CFG["use_fixes"] and not os.path.exists(LABEL_FIX_PATH):
    print(f"Không thấy {LABEL_FIX_PATH}: coi như nhãn đã được sửa sẵn trong train/val, bỏ qua bước áp nhãn sửa.")
elif CFG["use_fixes"]:
    fixes = pd.read_csv(LABEL_FIX_PATH).drop_duplicates(["split", "row_id"], keep="last")
    if (fixes["split"] == "test").any(): raise ValueError("Không sửa nhãn tập test bằng file này – test phải giữ nguyên để so sánh công bằng.")
    for name, d in [("train", train_df), ("val", val_df)]:
        f = fixes[fixes["split"] == name]
        idx = d.index[d["row_id"].isin(f["row_id"])]
        m = f.set_index("row_id")
        rid = d.loc[idx, "row_id"]
        if len(idx) and (d.loc[idx, "label_str"].values == m.loc[rid, "old_label"].values).mean() < 1.0:
            if (d.loc[idx, "label_str"].values == m.loc[rid, "new_label"].values).mean() == 1.0:
                print(f"{name}: các nhãn này đã được sửa sẵn trong file dữ liệu, bỏ qua"); continue
            raise ValueError(f"{name}: old_label trong label_fixes.csv không khớp dữ liệu – có thể sai phiên bản file")
        d.loc[idx, "label"] = m.loc[rid, "new_label"].map(LABEL_TO_ID).values
        print(f"{name}: đã sửa {len(idx)} nhãn")

for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    vc = d["label"].value_counts().reindex([0, 1, 2]).fillna(0).astype(int)
    print(f"{n:5s} n={len(d):6d} | clean={vc[0]} offensive={vc[1]} hate={vc[2]} | có quote={(d['quoted_text'].str.strip() != '').mean():.1%}")
fp = {n: hashlib.md5("\n".join(d["text"] + "||" + d["label"].astype(str)).encode()).hexdigest()[:12]
      for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
print("Fingerprint (test phải giống nhau giữa mọi giai đoạn):", fp)

## 2. Tokenize (ViHateT5 đọc văn bản gốc, không tách từ)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
PAD_ID = tokenizer.pad_token_id

def build_input(t, q=""):
    if INPUT_MODE == "text_quote" and q.strip():
        return f"{TASK_PREFIX}{t} </s> đang trả lời: {q}"
    return f"{TASK_PREFIX}{t}"

def encode(t, q=""):
    return tokenizer(build_input(t, q), max_length=MAX_LEN, truncation=True)["input_ids"]

TARGET_IDS = [tokenizer(x)["input_ids"] for x in TARGETS]      # mỗi nhãn kèm </s>
print("Token của các nhãn:", {x: tokenizer.convert_ids_to_tokens(i) for x, i in zip(TARGETS, TARGET_IDS)})

# ---- Nạp trọng số ViHateT5 ----
# Repo tarudesu/ViHateT5-base chỉ có flax_model.msgpack (không có pytorch_model.bin / model.safetensors),
# nên from_pretrained thường báo OSError. Khi đó đọc thẳng file msgpack và đổi sang state_dict của PyTorch
# (không cần cài jax/flax, và không phụ thuộc phiên bản transformers).
from transformers import AutoConfig
_USE_FLAX, _T5_SD = None, None

def _read_flax_msgpack(path):
    try:
        import msgpack
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "msgpack"]); import msgpack
    def hook(code, data):
        if code == 1:                                  # mảng numpy: (shape, tên dtype, bytes)
            shape, dtype, buf = msgpack.unpackb(data, raw=False)
            if dtype == "bfloat16": raise RuntimeError("file msgpack lưu bfloat16 – báo lại để đổi cách đọc")
            return np.frombuffer(buf, dtype=np.dtype(dtype)).reshape(shape)
        if code == 3:                                  # số vô hướng numpy
            dtype, buf = msgpack.unpackb(data, raw=False)
            return np.frombuffer(buf, dtype=np.dtype(dtype))[0]
        return msgpack.ExtType(code, data)
    with open(path, "rb") as f:
        return msgpack.unpackb(f.read(), ext_hook=hook, raw=False, strict_map_key=False)

def _flax_to_torch(tree):
    if list(tree.keys()) == ["params"]: tree = tree["params"]
    sd = {}
    def walk(node, path):
        if isinstance(node, dict):
            for k, v in node.items(): walk(v, path + [str(k)])
            return
        arr = np.asarray(node); name = ".".join(path)
        if path[-1] == "kernel":                       # Dense: flax (vào, ra) -> torch (ra, vào)
            name = name[:-len("kernel")] + "weight"; arr = arr.T
        elif path[-1] == "embedding":
            name = name[:-len("embedding")] + "weight"
        sd[name] = torch.from_numpy(np.ascontiguousarray(arr)).float()
    walk(tree, [])
    return sd

def load_pretrained_t5():
    """Trả về một mô hình ViHateT5 mới (trọng số tiền huấn luyện) cho mỗi lần gọi."""
    global _USE_FLAX, _T5_SD
    if _USE_FLAX is None:
        try:
            m = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME); _USE_FLAX = False; return m
        except OSError as e:
            _USE_FLAX = True; print("Không có trọng số PyTorch trong repo -> nạp từ flax_model.msgpack |", str(e)[:100])
    if _USE_FLAX is False:
        return AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    if _T5_SD is None:
        from huggingface_hub import hf_hub_download
        t0 = time.time()
        _T5_SD = _flax_to_torch(_read_flax_msgpack(hf_hub_download(MODEL_NAME, "flax_model.msgpack")))
        print(f"Đã đổi {len(_T5_SD)} tensor flax -> torch trong {time.time()-t0:.0f}s")
    cfg = AutoConfig.from_pretrained(MODEL_NAME)
    m = AutoModelForSeq2SeqLM.from_config(cfg)
    sd = dict(_T5_SD)
    for k in ("encoder.embed_tokens.weight", "decoder.embed_tokens.weight"): sd[k] = sd["shared.weight"]
    miss, unexp = m.load_state_dict(sd, strict=False)   # sai kích thước tensor sẽ tự báo lỗi
    miss = [k for k in miss if not (k == "lm_head.weight" and cfg.tie_word_embeddings)]
    if miss or unexp: raise RuntimeError(f"Đổi trọng số chưa khớp. Thiếu: {miss[:5]} | Thừa: {list(unexp)[:5]}")
    m.tie_weights()
    return m

def pad(seqs, value):
    L = max(len(s) for s in seqs)
    out = torch.full((len(seqs), L), value, dtype=torch.long)
    for i, s in enumerate(seqs): out[i, :len(s)] = torch.tensor(s)
    return out

def collate(batch):
    ids = pad([b["ids"] for b in batch], PAD_ID)
    return {"input_ids": ids, "attention_mask": (ids != PAD_ID).long(),
            "labels_seq": pad([TARGET_IDS[b["y"]] for b in batch], -100), "labels": torch.tensor([b["y"] for b in batch])}

class CommentDataset(Dataset):
    def __init__(self, frame):
        self.ids = [encode(t, q) for t, q in zip(frame["text"], frame["quoted_text"])]
        self.y = frame["label"].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return {"ids": self.ids[i], "y": self.y[i]}

t0 = time.time()
train_ds, val_ds, test_ds = CommentDataset(train_df), CommentDataset(val_df), CommentDataset(test_df)
lens = [len(x) for x in train_ds.ids]
print(f"Tokenize {time.time()-t0:.0f}s | token trung vị {int(np.median(lens))}, p95 {int(np.percentile(lens, 95))}, bị cắt {np.mean([l >= MAX_LEN for l in lens]):.1%}")
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate)
y_val, y_test = val_df["label"].values, test_df["label"].values
has_q_test = (test_df["quoted_text"].str.strip() != "").values

## 3. Chấm điểm 3 nhãn bằng log-likelihood
Encoder chạy một lần; decoder chạy một lần cho mỗi nhãn với teacher forcing. Điểm của nhãn = tổng log-xác suất các token của chuỗi nhãn.

In [ ]:
def mv(b): return {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}

@torch.no_grad()
def label_scores(model, input_ids, attention_mask, amp=None):
    amp = use_amp if amp is None else amp
    dev = input_ids.device; B = input_ids.size(0)
    with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):
        enc = model.get_encoder()(input_ids=input_ids, attention_mask=attention_mask)
        scores = []
        for tgt in TARGET_IDS:
            lab = torch.tensor(tgt, device=dev).unsqueeze(0).expand(B, -1).contiguous()   # expand() cho tensor stride 0 -> T5 gọi labels.view(-1) sẽ lỗi
            out = model(encoder_outputs=enc, attention_mask=attention_mask, labels=lab)
            lp = F.log_softmax(out.logits.float(), -1).gather(-1, lab.unsqueeze(-1)).squeeze(-1)
            scores.append(lp.sum(-1))
    return torch.stack(scores, -1).float()           # (B, 3) – dùng như logits

@torch.no_grad()
def predict_logits(model, loader):
    model.eval(); out = []
    for b in loader:
        b = mv(b); out.append(label_scores(model, b["input_ids"], b["attention_mask"]).cpu())
    return torch.cat(out).numpy()

def macro_f1(y, p): return f1_score(y, p, labels=[0, 1, 2], average="macro", zero_division=0)

## 4. Hiệu chỉnh và các chỉ số cho cascade (giống hệt notebook PhoBERT)

In [ ]:
def softmax(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def fit_temperature(logits, y):
    """Chọn T > 0 cực tiểu NLL trên tập val (Guo et al. 2017)."""
    def nll(logT):
        p = softmax(logits / np.exp(logT)); return -np.log(p[np.arange(len(y)), y] + 1e-12).mean()
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

def ece(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def nll(probs, y): return float(-np.log(probs[np.arange(len(y)), y] + 1e-12).mean())

def coverage_at_acc(p_val, yv, p_test, yt, target=TARGET_ACC):
    """Chọn τ nhỏ nhất trên VAL sao cho độ chính xác phần tự quyết >= target, rồi đo trên TEST."""
    cv, pv = p_val.max(1), p_val.argmax(1); tau = None
    for t in np.round(np.arange(0.50, 0.996, 0.005), 3):
        m = cv >= t
        if m.sum() >= 20 and (pv[m] == yv[m]).mean() >= target: tau = float(t); break
    if tau is None: return {"tau": None, "coverage": 0.0, "acc_decided": float("nan")}
    ct, pt = p_test.max(1), p_test.argmax(1); m = ct >= tau
    return {"tau": tau, "coverage": float(m.mean()), "acc_decided": float((pt[m] == yt[m]).mean()) if m.any() else float("nan")}

def all_metrics(seed, best, lv, lt):
    T = fit_temperature(lv, y_val)
    p_raw, p_cal, pv_cal = softmax(lt), softmax(lt / T), softmax(lv / T)
    pred = lt.argmax(1); pc = f1_score(y_test, pred, labels=[0, 1, 2], average=None, zero_division=0)
    cov = coverage_at_acc(pv_cal, y_val, p_cal, y_test)
    return {"stage": STAGE, "tag": TAG, "seed": seed, "best_epoch": best["epoch"], "val_macro_f1": best["f1"],
            "test_macro_f1": macro_f1(y_test, pred), "test_acc": accuracy_score(y_test, pred),
            "f1_clean": pc[0], "f1_offensive": pc[1], "f1_hate": pc[2],
            "macro_f1_has_quote": macro_f1(y_test[has_q_test], pred[has_q_test]),
            "macro_f1_no_quote": macro_f1(y_test[~has_q_test], pred[~has_q_test]),
            "temperature": T, "ece_raw": ece(p_raw, y_test), "ece_cal": ece(p_cal, y_test),
            "nll_raw": nll(p_raw, y_test), "nll_cal": nll(p_cal, y_test),
            "tau2": cov["tau"], "coverage_tier2": cov["coverage"], "acc_decided_tier2": cov["acc_decided"]}

## 5. Huấn luyện

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def train_one_seed(seed):
    set_seed(seed)
    gen = torch.Generator(); gen.manual_seed(seed)
    tr_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=gen, collate_fn=collate, pin_memory=DEVICE.type == "cuda")
    m = load_pretrained_t5().to(DEVICE)
    no_decay = ("bias", "layer_norm.weight")
    groups = [{"params": [p for n, p in m.named_parameters() if not n.endswith(no_decay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in m.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=LR)
    steps = len(tr_dl) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    best = {"f1": -1, "epoch": 0, "state": None}; hist = []; bad = 0
    for ep in range(1, EPOCHS + 1):
        m.train(); t0 = time.time(); run = 0.0; seen = 0
        for step, b in enumerate(tr_dl, 1):
            b = mv(b); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                loss = m(input_ids=b["input_ids"], attention_mask=b["attention_mask"], labels=b["labels_seq"]).loss
            if not torch.isfinite(loss): raise RuntimeError("loss = NaN/inf – nếu đang bật USE_FP16 thì tắt đi")
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            run += loss.item() * len(b["labels"]); seen += len(b["labels"])
            if step % 200 == 0: print(f"  seed {seed} ep {ep} step {step}/{len(tr_dl)} loss={run/seen:.4f}", flush=True)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        train_sec = time.time() - t0
        lv = predict_logits(m, val_dl)
        vf1 = macro_f1(y_val, lv.argmax(1))
        vloss = float(F.cross_entropy(torch.tensor(lv), torch.tensor(y_val)).item())
        hist.append({"seed": seed, "epoch": ep, "train_loss": run / seen, "val_nll": vloss, "val_macro_f1": vf1,
                     "minutes": (time.time() - t0) / 60, "train_samples_per_s": seen / train_sec,
                     "peak_vram_train_mb": torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else None})
        print(f"[seed {seed} | epoch {ep}] train_loss={run/seen:.4f} val_nll={vloss:.4f} val_macro_f1={vf1:.4f} ({hist[-1]['minutes']:.1f} phút)", flush=True)
        if vf1 > best["f1"]:
            best = {"f1": vf1, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}}; bad = 0
        else:
            bad += 1
            if CFG["patience"] is not None and bad >= CFG["patience"]:
                print(f"  Dừng sớm ở epoch {ep}"); break
    m.load_state_dict(best["state"])
    best["train_samples_per_s"] = float(np.mean([h["train_samples_per_s"] for h in hist]))
    best["peak_vram_train_mb"] = max((h["peak_vram_train_mb"] or 0) for h in hist) if DEVICE.type == "cuda" else None
    best["epochs_run"] = len(hist)
    return m, best, hist

## 6. Chạy các seed

In [ ]:
RES_PATH = os.path.join(OUT_DIR, f"results_{TAG}.csv"); HIST_PATH = os.path.join(OUT_DIR, f"history_{TAG}.csv")
BEST_DIR = os.path.join(OUT_DIR, f"{TAG}_best")
results, histories, best_overall = [], [], {"val": -1}
for seed in SEEDS:
    t0 = time.time()
    m, best, hist = train_one_seed(seed)
    lv, lt = predict_logits(m, val_dl), predict_logits(m, test_dl)
    np.savez_compressed(os.path.join(OUT_DIR, f"logits_{TAG}_seed{seed}.npz"), val=lv, test=lt, y_val=y_val, y_test=y_test)
    r = all_metrics(seed, best, lv, lt); r["train_minutes"] = (time.time() - t0) / 60
    r.update({"backbone": BACKBONE, "epochs_run": best["epochs_run"], "train_samples_per_s": best["train_samples_per_s"],
              "peak_vram_train_mb": best["peak_vram_train_mb"]})
    results.append(r); histories += hist
    pd.DataFrame(results).to_csv(RES_PATH, index=False); pd.DataFrame(histories).to_csv(HIST_PATH, index=False)
    print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}, ensure_ascii=False))
    if best["f1"] > best_overall["val"]:
        best_overall = {"val": best["f1"], "seed": seed}
        m.save_pretrained(BEST_DIR); tokenizer.save_pretrained(BEST_DIR)
        json.dump({"temperature": r["temperature"], "threshold": r["tau2"] if r["tau2"] is not None else 0.99,
                   "task_prefix": TASK_PREFIX, "targets": TARGETS, "input_mode": INPUT_MODE, "max_len": MAX_LEN, "seed": seed},
                  open(os.path.join(BEST_DIR, "calibration.json"), "w"), indent=2, ensure_ascii=False)
    del m; gc.collect(); torch.cuda.empty_cache()
print("Mô hình tốt nhất:", BEST_DIR, "| seed", best_overall["seed"])

## 7. Tổng kết mean ± std

In [ ]:
res = pd.DataFrame(results)
cols = ["val_macro_f1", "test_macro_f1", "test_acc", "f1_clean", "f1_offensive", "f1_hate", "macro_f1_has_quote",
        "macro_f1_no_quote", "ece_raw", "ece_cal", "nll_raw", "nll_cal", "temperature", "coverage_tier2", "acc_decided_tier2",
        "best_epoch", "epochs_run", "train_minutes", "train_samples_per_s", "peak_vram_train_mb"]
summary = pd.DataFrame({"mean": res[cols].astype(float).mean(), "std": res[cols].astype(float).std(ddof=1)})
summary["mean ± std"] = [f"{a:.4f} ± {b:.4f}" for a, b in zip(summary["mean"], summary["std"])]
display(summary[["mean ± std"]])
json.dump({"stage": STAGE, "tag": TAG, "cfg": CFG, "n_seeds": len(res),
           **{c: {"mean": float(summary.loc[c, "mean"]), "std": float(summary.loc[c, "std"])} for c in cols}},
          open(os.path.join(OUT_DIR, f"summary_{TAG}.json"), "w"), indent=2, ensure_ascii=False)
h = pd.DataFrame(histories)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for s, g in h.groupby("seed"):
    ax[0].plot(g["epoch"], g["val_nll"], marker="o", label=f"seed {s}"); ax[1].plot(g["epoch"], g["val_macro_f1"], marker="o", label=f"seed {s}")
ax[0].set_title("Val NLL theo epoch"); ax[1].set_title("Val Macro-F1 theo epoch")
for a in ax: a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f"curves_{TAG}.png"), dpi=150); plt.show()

## 8. Đo tốc độ suy luận (cùng tên cột với notebook PhoBERT)
ViHateT5 không cần tách từ (`gpu_e2e_segment_ms` = 0), nhưng mỗi bình luận cần 1 lần encoder + 3 lần decoder.

In [ ]:
m = AutoModelForSeq2SeqLM.from_pretrained(BEST_DIR).to(DEVICE).eval()
amp_inf = DEVICE.type == "cuda"     # suy luận fp16 thường ổn hơn train; nếu thấy kết quả lệch thì đặt False
if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
def sync(dev):
    if dev.type == "cuda": torch.cuda.synchronize()

def time_b1(model, dev, items, amp):
    lat = []
    for k, b in enumerate(items):
        x = collate([b]); ids, am = x["input_ids"].to(dev), x["attention_mask"].to(dev)
        sync(dev); t = time.perf_counter(); label_scores(model, ids, am, amp).argmax(-1); sync(dev)
        if k >= 10: lat.append((time.perf_counter() - t) * 1000)
    return float(np.mean(lat)), float(np.percentile(lat, 95))

def time_throughput(model, dev, ds, amp, bs=64):
    dl = DataLoader(ds, batch_size=bs, collate_fn=collate); n = 0
    for k, b in enumerate(dl):
        if k == 1: sync(dev); t = time.perf_counter(); n = 0
        label_scores(model, b["input_ids"].to(dev), b["attention_mask"].to(dev), amp); n += len(b["labels"])
    sync(dev); return n / (time.perf_counter() - t)

def time_end_to_end(model, dev, texts, quotes, amp):
    tok_ms, mdl_ms = [], []
    for k, (t_, q_) in enumerate(zip(texts, quotes)):
        a = time.perf_counter(); x = collate([{"ids": encode(t_, q_), "y": 0}])
        ids, am = x["input_ids"].to(dev), x["attention_mask"].to(dev)
        b_ = time.perf_counter(); sync(dev); label_scores(model, ids, am, amp).argmax(-1).item(); sync(dev); c_ = time.perf_counter()
        if k >= 10: tok_ms.append((b_ - a) * 1000); mdl_ms.append((c_ - b_) * 1000)
    tot = np.array(tok_ms) + np.array(mdl_ms)
    return {"segment_ms": 0.0, "tokenize_ms": float(np.mean(tok_ms)), "model_ms": float(np.mean(mdl_ms)),
            "total_ms": float(tot.mean()), "total_p95_ms": float(np.percentile(tot, 95))}

N_B1, N_E2E, N_CPU = 310, 210, 110
items = [test_ds[i] for i in range(N_B1)]
raw_t, raw_q = test_df["text"].tolist(), test_df["quoted_text"].tolist()
speed = {"tag": TAG, "backbone": BACKBONE, "model_name": MODEL_NAME, "head": "seq2seq-score", "needs_seg": False,
         "input_mode": INPUT_MODE, "max_len": MAX_LEN, "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
         "cpu": f"{platform.processor() or platform.machine()} x{os.cpu_count()} threads",
         "params_million": sum(p.numel() for p in m.parameters()) / 1e6,
         "ckpt_mb": sum(os.path.getsize(os.path.join(BEST_DIR, f)) for f in os.listdir(BEST_DIR)) / 2**20}
if DEVICE.type == "cuda":
    speed["gpu_b1_dynamic_ms"], speed["gpu_b1_dynamic_p95_ms"] = time_b1(m, DEVICE, items, amp_inf)
    speed["gpu_b64_dynamic_per_s"] = time_throughput(m, DEVICE, test_ds, amp_inf)
    speed.update({f"gpu_e2e_{k}": v for k, v in time_end_to_end(m, DEVICE, raw_t[:N_E2E], raw_q[:N_E2E], amp_inf).items()})
    speed["gpu_peak_vram_infer_mb"] = torch.cuda.max_memory_allocated() / 2**20
    speed["gpu_peak_vram_train_mb"] = float(pd.DataFrame(results)["peak_vram_train_mb"].max())
    speed["train_samples_per_s"] = float(pd.DataFrame(results)["train_samples_per_s"].mean())
    # kiểm tra fp16 khi suy luận không làm đổi dự đoán so với fp32
    sub = DataLoader(torch.utils.data.Subset(test_ds, range(256)), batch_size=32, collate_fn=collate)
    agree = np.mean(np.concatenate([(label_scores(m, b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE), True).argmax(-1)
                                     == label_scores(m, b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE), False).argmax(-1)).cpu().numpy() for b in sub]))
    speed["fp16_vs_fp32_agreement"] = float(agree); print(f"fp16 và fp32 cho cùng dự đoán ở {agree:.1%} mẫu")
if MEASURE_CPU:
    cpu = torch.device("cpu"); torch.set_num_threads(os.cpu_count()); mc = m.float().to(cpu).eval()
    speed["cpu_b1_dynamic_ms"], speed["cpu_b1_dynamic_p95_ms"] = time_b1(mc, cpu, items[:N_CPU], False)
    speed.update({f"cpu_e2e_{k}": v for k, v in time_end_to_end(mc, cpu, raw_t[:N_CPU], raw_q[:N_CPU], False).items()})
print(json.dumps({k: (round(v, 2) if isinstance(v, float) else v) for k, v in speed.items()}, indent=2, ensure_ascii=False))
json.dump(speed, open(os.path.join(OUT_DIR, f"speed_{TAG}.json"), "w"), indent=2, ensure_ascii=False)

## 9. So sánh với các mô hình khác
Chép các file `results_*.csv` và `speed_*.json` của notebook PhoBERT vào một dataset Kaggle rồi thêm đường dẫn vào `EXTRA_DIRS`.

In [ ]:
import glob
EXTRA_DIRS = []    # ví dụ ["/kaggle/input/ket-qua-stage0"]
files = sorted({f for d in [OUT_DIR] + EXTRA_DIRS for f in glob.glob(os.path.join(d, "results_*.csv"))})
if files:
    allr = pd.concat([pd.read_csv(f) for f in files])
    show = ["test_macro_f1", "f1_offensive", "f1_hate", "macro_f1_has_quote", "ece_raw", "ece_cal", "coverage_tier2"]
    g = allr.groupby("tag")[show]
    table = g.mean().round(4).astype(str) + " ± " + g.std(ddof=1).round(4).astype(str)
    table.insert(0, "n_seed", allr.groupby("tag").size())
    sp_files = sorted({f for d in [OUT_DIR] + EXTRA_DIRS for f in glob.glob(os.path.join(d, "speed_*.json"))})
    if sp_files:
        sp = pd.DataFrame([json.load(open(f)) for f in sp_files]).set_index("tag")
        keep = [c for c in ["params_million", "gpu_b1_dynamic_ms", "gpu_e2e_total_ms", "gpu_b64_dynamic_per_s", "cpu_e2e_total_ms",
                            "train_samples_per_s", "gpu_peak_vram_train_mb"] if c in sp.columns]
        table = table.join(sp[keep].round(1), how="left")
    display(table); table.to_csv(os.path.join(OUT_DIR, "compare_stages.csv"), encoding="utf-8-sig")

## Ghi chú
- Chạy notebook này trên **cùng dataset** (đã sửa nhãn hay chưa) với giai đoạn PhoBERT đem ra so sánh, và giữ nguyên tập test.
- Nếu ba seed mất quá lâu, có thể bật `USE_FP16 = True`; khi đó nhìn dòng `loss` – nếu ra NaN thì tắt lại.
- Trích dẫn: Luan Thanh Nguyen. *ViHateT5: Enhancing Hate Speech Detection in Vietnamese with a Unified Text-to-Text Transformer Model.* Findings of ACL 2024.

## Gói kết quả để tải về
In danh sách file đã ghi và nén các file nhỏ (csv, json, png, npz) thành `results_bundle.zip`. Checkpoint (`*_best.pt` hoặc thư mục `*_best`) rất nặng nên để riêng.

In [ ]:
import zipfile, glob
print("Thư mục kết quả:", OUT_DIR)
rows = []
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*"))):
    if os.path.isfile(f): rows.append((os.path.basename(f), os.path.getsize(f) / 2**20))
    elif os.path.isdir(f) and f.endswith("_best"): rows.append((os.path.basename(f) + "/", sum(os.path.getsize(os.path.join(f, x)) for x in os.listdir(f)) / 2**20))
print(pd.DataFrame(rows, columns=["file", "MB"]).round(2).to_string(index=False))
small = [f for f in glob.glob(os.path.join(OUT_DIR, "*")) if os.path.isfile(f) and f.endswith((".csv", ".json", ".png", ".npz")) and "seg_cache" not in f]
ZIP_PATH = os.path.join(OUT_DIR, "results_bundle.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in small: z.write(f, os.path.basename(f))
print(f"\\nĐã nén {len(small)} file vào {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/2**20:.1f} MB)")
try:
    from google.colab import files; files.download(ZIP_PATH)      # chỉ chạy trên Colab
except ImportError:
    pass